# Metadata Filtering Notebook

This notebook filters the NIH ChestXray14 metadata to retain only the records for images that have been successfully extracted and are available locally in the `data/images/` directory.

In [ ]:
import os
import pandas as pd
from glob import glob

# Paths
DATA_DIR = 'data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
RAW_METADATA_PATH = os.path.join(DATA_DIR, 'Data_Entry_2017_v2020.csv')
FILTERED_METADATA_PATH = os.path.join(DATA_DIR, 'metadata_filtered.csv')

In [ ]:
# Step 1: Scan locally available images
if os.path.exists(IMAGES_DIR):
    print("Scanning locally available images in", IMAGES_DIR, "...")
    local_images = set(os.listdir(IMAGES_DIR))
    print(f"Total unique images found locally: {len(local_images)}")
else:
    print(f"Error: {IMAGES_DIR} directory does not exist. Please run Data_Pipeline.ipynb Step 2 first.")
    local_images = set()

In [ ]:
# Step 2: Load and Filter Metadata
if os.path.exists(RAW_METADATA_PATH) and len(local_images) > 0:
    df_raw = pd.read_csv(RAW_METADATA_PATH)
    print(f"Original Metadata Records: {len(df_raw)}")
    
    # Filter: Keep only rows where 'Image Index' exists in our local folder
    df_filtered = df_raw[df_raw['Image Index'].isin(local_images)].reset_index(drop=True)
    
    # Keeping only essential columns
    required_columns = ['Image Index', 'Finding Labels']
    df_final = df_filtered[required_columns]
    
    # Save the filtered metadata
    df_final.to_csv(FILTERED_METADATA_PATH, index=False)
    print(f"Filtered Metadata Records: {len(df_final)}")
    print(f"Saved to {FILTERED_METADATA_PATH}")
elif len(local_images) == 0:
    print("No images found to filter.")
else:
    print(f"Error: Raw metadata not found at {RAW_METADATA_PATH}")

In [ ]:
# Step 3: Prove Load
if 'df_final' in locals():
    print("--- Filtered Metadata Preview (Only Extracted Images) ---")
    display(df_final.head(10))
    
    print("\n--- Statistics ---")
    print(f"Images In Folder: {len(local_images)}")
    print(f"Rows in Filtered Metadata: {len(df_final)}")
    
    if len(local_images) == len(df_final):
        print("\nSUCCESS: Metadata matches local storage perfectly.")
    else:
        print("\nNOTE: Some images in folder might not be in metadata or vice versa.")